#### Question 1 
Remove the Duplicates and only keep latest record!

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
data=[
(101,'Anurag','2026-04-03'),
(101,'Anurag','2026-05-03'),
(102,'Stuti','2026-05-06'),
(102,'Stuti','2026-06-14'),
(103,'Ravi','2026-07-17')
]

In [0]:
columns = ["Customer_id","Customer_name","Update_date"]

In [0]:
df = spark.createDataFrame(data,columns)
df = df.withColumn("update_date",F.to_date("update_date","yyyy-MM-dd"))

In [0]:
display(df)

In [0]:
display(df.dropDuplicates( ))
display(df.dropDuplicates(['Customer_id']))
display(df.dropDuplicates(['Customer_id','customer_name']))

In [0]:
window = Window.partitionBy("customer_id").orderBy(F.col("update_date").desc())

df = df.withColumn("rank", F.dense_rank().over(window)).where("rank = 1").drop("rank")
display(df)

In [0]:
print(window)

##### Question 2 : Find Highest Salaried Employee Per Department

In [0]:
employee_data = [
    (1, "Anurag", "IT", 90000),
    (2, "Shalini", "IT", 85000),
    (3, "Stuti", "HR", 70000),
    (4, "Ritika", "HR", 70000),
    (5, "Vaishnav", "Finance", 95000)
]

employee_columns = [
    "emp_id",
    "emp_name",
    "department",
    "salary"
]

employee_df = spark.createDataFrame(employee_data, employee_columns)
window = Window.partitionBy("department").orderBy(F.col("salary").desc())
employee_df1 = employee_df.withColumn("rank",F.dense_rank().over(window)).where("rank = 1").drop("rank")
display(employee_df)
display(employee_df1)

##### Question 3: Month-over-Month Revenue Growth

In [0]:
sales_data = [
    (101, "2024-01", 10000),
    (101, "2024-02", 15000),
    (101, "2024-03", 12000),
    (102, "2024-01", 8000),
    (102, "2024-02", 9000),
    (102, "2024-03", 11000)
]

sales_columns = [
    "customer_id",
    "month",
    "revenue"
]

df = spark.createDataFrame(sales_data, sales_columns).withColumn("month",(F.to_date("month","yyyy-MM")))
#display(df)
window = Window.partitionBy("customer_id").orderBy(F.col("month")).rowsBetween(Window.unboundedPreceding,Window.currentRow)
df1 = df.withColumn("cummulative_sum",F.sum("revenue").over(window))
display(df1)

window1 = Window.partitionBy("customer_id").orderBy(F.col("month"))
df2 = df.withColumn("previous_month_revenue",F.coalesce(F.lag("revenue").over(window1),F.lit(0))).withColumn("revenue_growth",F.col("revenue")-F.col("previous_month_revenue"))
display(df2)


##### Question 4: Optimize a Join


In [0]:
transaction_data = [
    (1, 101, "IN", 5000),
    (2, 102, "US", 7000),
    (3, 103, "IN", 3000),
    (4, 104, "UK", 9000)
]

country_data = [
    ("IN", "India"),
    ("US", "United States"),
    ("UK", "United Kingdom")
]

transaction_df = spark.createDataFrame(transaction_data,['txn_id','cust_id','country_code','amount'])
country_df=spark.createDataFrame(country_data,['country_code','country_name'])
display(transaction_df)
display(country_df)

In [0]:
df_join = transaction_df.join(F.broadcast(country_df),"country_code")
display(df_join)


##### Question 5: Incremental Loading

In [0]:
orders_data = [
    (1, "Created", "2024-01-01"),
    (2, "Shipped", "2024-01-05"),
    (3, "Delivered", "2024-01-10"),
    (4, "Cancelled", "2024-01-15")
]

last_processed_date='2024-01-10'
orders_df=spark.createDataFrame(orders_data,['order_id','status','updated_at']).withColumn("updated_at",F.to_date("updated_at","yyyy-MM-dd"))

In [0]:
incremental_df=orders_df.filter(F.col("updated_at")>last_processed_date)
display(incremental_df)

#### Question 1
### Flatten the Nested Order Data

In [0]:
from pyspark.sql.types import *

In [0]:
customer_data = [
    (
        101,
        "Anurag",
        [
            {"order_id": 1, "product": "Laptop", "amount": 50000},
            {"order_id": 2, "product": "Mouse", "amount": 1000}
        ]
    ),
    (
        102,
        "Stuti",
        [
            {"order_id": 3, "product": "Keyboard", "amount": 2500}
        ]
    ),
    (
        103,
        "Rahul",
        []
    )
]


In [0]:
order_schema = ArrayType(
    StructType([
                          StructField("order_id", IntegerType(), True),
                          StructField("product", StringType(), True),
                          StructField("amount", IntegerType(), True)
                ])
)

customer_schema = StructType([
                            StructField("customer_id", IntegerType(), True),
                            StructField("customer_name", StringType(), True),
                            StructField("orders", order_schema, True)
                            ])



In [0]:
customer_df= spark.createDataFrame(customer_data,customer_schema)
display(customer_df)

### Explode the Array

In [0]:
exploded_df = customer_df.withColumn("orders_exploded",F.explode_outer("orders")).drop("orders")
display(exploded_df)

In [0]:
final_orders_df = exploded_df.select("*","orders_exploded.*").drop("orders_exploded")
display(final_orders_df)

#### Question 2
#### Union DataFrames with Different Column Orders
You are getting daily files in that the columns are same but the order of the columns are different 

so how you will safely combine them?

In [0]:
day1_data = [
    (1, "Anurag", 50000),
    (2, "Stuti", 60000)
]
day2_data = [
    (70000, 3, "Shruti"),
    (80000, 4, "Shalini")
]
day1_data_df=spark.createDataFrame(day1_data,["emp_id","emp_name","salary"])
day2_data_df=spark.createDataFrame(day2_data,["salary","emp_id","emp_name"])

display(day1_data_df)
display(day2_data_df)

In [0]:
df_union = day1_data_df.unionByName(day2_data_df)
display(df_union)

In [0]:
df_union2 = day1_data_df.union(day2_data_df.select("emp_id","emp_name","salary"))
display(df_union2)

##### If there is a scenario where there is a new Dataframe having missing columns

In [0]:
day3_data = [
    (5, "Vaishnav", 90000, "IT")
]

day3_df = spark.createDataFrame(
    day3_data,
    ["emp_id", "emp_name", "salary", "department"]
)
display(day3_df)

In [0]:
#allowMissingColumns=True
combined_df=day3_df.unionByName(df_union,allowMissingColumns=True)
display(combined_df)
# unionByName() behaves like SQL UNION ALL, meaning it does not remove duplicate rows automatically. 
# You must chain .distinct() to the result if you need unique records.


**Question1 : Two Source Records match one taget record**


- Target :Customers 
 
    - Customer_id, City 
    - 101, Delhi
    - 102, Bengaluru
    - 103, Pune


- Source: Updates
 
    - Customer_id, City 
    - 101,Mumbai 
    - 101,Pune

In [0]:
%sql
Merge into target t 
using source s
on s.customer_id = t.cutomer_id
WHEN MATCHED THEN
    UPDATE SET 
        t.city = s.city
WHEN NOT MATCHED THEN
    INSERT (customer_id,city)
    VALUES (s.customer_id,s.city)

In [0]:
data1 = [(1, 30), (2, 30), (3, 35), (4, 35)]
df_age = spark.createDataFrame(data1, ["id", "age"])

data2 = [(1, 1000), (2, 2000), (3, 1900), (4, 1900)]
df_salary = spark.createDataFrame(data2, ["id", "salary"])

In [0]:
df = df_age.join(df_salary,"id","inner")

In [0]:
display(df)

In [0]:
result_df = (df.groupBy("age") 
    .agg(F.avg("salary").alias("avg_salary")) 
    .orderBy(F.col("avg_salary").desc()) 
    .limit(1)
)

In [0]:
window = Window.partitionBy("age")
sf = df.withColumn("avg_salary", F.avg("salary").over(window)).select()
display(sf)
df2 = df.withColumn("avg_salary", F.avg("salary").over(window)).select("age","avg_salary").dropDuplicates()
max_salary = df2.select(F.max("avg_salary")).collect()[0][0]
print(max_salary)
display(df2.filter(F.col("avg_salary") == max_salary))

In [0]:
display(result_df)

##### Question 1 : Tiger Analytics
###### From a list of strings, extract dates using basic string manipulation (not regex) 
###### and return only those with year > 2018.
lines = ["Order placed on 2017-05-12", "Delivery on 2019-11-01", "Check 2020-06-15"]

In [0]:
# my solution
lines = ["Order placed on 2017-05-12", "Delivery on 2019-11-01", "Check 2020-06-15"]
lines_split = []
for str in lines:
    for i in str.split():
        lines_split.append(i)

print(lines_split)
df = spark.createDataFrame(lines_split,schema= "line2 string")
df2 = df.withColumn("date_col",F.try_to_date(F.col("line2"),"yyyy-MM-dd")).drop("line2").where("date_col is not null and year(date_col)>2018")
display(df2)

In [0]:
# Given Solution
lines = ["Order placed on 2017-05-12", "Delivery on 2019-11-01", "Check 2020-06-15"]
valid_dates = []
for line in lines:
    for i in range(len(line) - 9):
        sub = line[i:i+10]
        if sub[4] == '-' and sub[7] == '-' and sub[:4].isdigit():
            year = int(sub[:4])
            if year > 2018:
                valid_dates.append(sub)
print(valid_dates)

##### Question 2 : Tiger Analytics
##### PySpark DQ Check — Valid Date Format
###### Use PySpark to filter out rows with invalid date formats (yyyy-MM-dd):



In [0]:
lst = ['2000-01-01','10-12-2022']
df = spark.createDataFrame(lst,schema="date_col string")

In [0]:
from pyspark.sql.functions import col, to_date

df_valid = df.filter(F.try_to_date(col("date_col"),"yyyy-MM-dd").isNotNull())

In [0]:
display(df_valid)

##### Question 3 : Tiger Analytics
##### PySpark DQ Check — Alphabet-Only Names
###### Ensure employee_name contains only alphabetic characters:



In [0]:
df_clean = df.filter(col("employee_name").rlike("^[A-Za-z]+$"))

##### Question 4 : Tiger Analytics
##### SCD Type 1 vs Type 2 in PySpark
###### 🔁 Type 1 — Overwrite with No History

In [0]:
df_updated = df_source.alias("src").join(df_target.alias("tgt"),"id","left").selectExpr("*")